# CoordinX - Final Three-Branch Model Training

Notebook ini menjalankan training final research-ready setelah screening Organic:

1. IndoBERTweet frozen encoder + account pooling.
2. Temporal LSTM untuk urutan post.
3. GraphSAGE untuk message passing pada graph akun.
4. Fusion 0.4 / 0.3 / 0.3.
5. Gold-only holdout evaluation, confusion matrix, review queue, dan artifact contract.

Notebook ini **belum mengubah `/web`**. Integrasi web hanya dimulai setelah readiness gate dan approval Phase terakhir.

In [1]:
from pathlib import Path
import json
import os
import sys

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "PRD.md").exists())
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib-cache")
print("Repository:", ROOT)
print("Web integration: intentionally disabled")

Repository: /home/andry-alfisi/Project/Project/Buzzer-Detection-
Web integration: intentionally disabled


In [2]:
from preprocessing.organic_screening import run_screening
from models.final_training import run_final_training

organic_summary = run_screening(ROOT)
print("Organic candidates:", organic_summary["organic_candidates"])
result = run_final_training(
    root=ROOT,
    test_size=0.30,
    lstm_epochs=80,
    graphsage_epochs=100,
)
readiness = result["readiness"]
print(json.dumps(readiness, indent=2, ensure_ascii=False, default=str))

/home/andry-alfisi/anaconda3/lib/python3.13/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


Organic candidates: 217


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: indolem/indobertweet-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


{
  "status": "RESEARCH_FINAL_READY_FOR_WEB_INTEGRATION",
  "generated_at_utc": "2026-11-11T13:43:05.652547+00:00",
  "model_version": "coordinx-final-v1.0.0",
  "schema_version": "coordinx-final-output-v1",
  "training_batches": [
    "batch2",
    "batch3",
    "batch4",
    "batch5"
  ],
  "batch6_policy": "excluded because it overlaps batch4 and is not an independent issue",
  "accounts_total": 3689,
  "gold_labels": 200,
  "weak_labels": 217,
  "gold_class_counts": {
    "0": 49,
    "1": 151
  },
  "weak_class_counts": {
    "0": 217
  },
  "gold_holdout": {
    "test_accounts": 60,
    "metrics": [
      {
        "model": "IndoBERTweet",
        "n": 60,
        "accuracy": 0.8166666666666667,
        "precision_buzzer": 0.9047619047619048,
        "recall_buzzer": 0.8444444444444444,
        "f1_buzzer": 0.8735632183908046,
        "confusion_matrix": [
          [
            11,
            4
          ],
          [
            7,
            38
          ]
        ],
     

In [3]:
metrics = result["metrics"]
predictions = result["predictions"]
metrics[["model", "n", "accuracy", "precision_buzzer", "recall_buzzer", "f1_buzzer", "roc_auc", "pr_auc"]]

,model,n,accuracy,precision_buzzer,recall_buzzer,f1_buzzer,roc_auc,pr_auc
0,IndoBERTweet,60,0.816667,0.904762,0.844444,0.873563,0.865185,0.955437
1,Temporal LSTM,60,0.783333,0.833333,0.888889,0.860215,0.807407,0.917662
2,GraphSAGE,60,0.866667,0.911111,0.911111,0.911111,0.854815,0.911411
3,Fusion,60,0.833333,0.906977,0.866667,0.886364,0.897778,0.967104


In [4]:
required_columns = {
    "account_id", "username", "dataset_id", "issue_id", "prediction",
    "coordination_score", "probability_buzzer", "confidence", "risk_level",
    "indobertweet_probability", "temporal_lstm_probability",
    "graphsage_probability", "explanation", "review_status", "model_version",
}
assert readiness["status"] == "RESEARCH_FINAL_READY_FOR_WEB_INTEGRATION"
assert readiness["accounts_total"] == len(predictions)
assert required_columns.issubset(predictions.columns)
assert set(predictions["prediction"]).issubset({"Buzzer", "Organic", "Review"})
assert len(metrics) == 4
for path in [
    ROOT / "models/final/indobertweet_final.pkl",
    ROOT / "models/final/temporal_lstm_final.pt",
    ROOT / "models/final/graphsage_final.pt",
    ROOT / "output/final_model/final_predictions.csv",
    ROOT / "output/final_model/final_gold_holdout_metrics.csv",
    ROOT / "output/final_model/final_readiness.json",
]:
    assert path.exists(), path
print("Final model notebook validation: OK")
print("Prediction counts:", predictions["prediction"].value_counts().to_dict())

Final model notebook validation: OK
Prediction counts: {'Buzzer': 1981, 'Organic': 1379, 'Review': 329}
